# Bangladesh: strictly sequential execution

Local workspace: `D:\Gihub_Projects\BD_GEE_Analysis`.

Run cells in order. `tutorials` selects all non-deferred individual examples; `national` selects the 16 improved national workflows. Never launch both simultaneously. Read `docs/SEQUENTIAL.md` first. No remote execution was verified during preparation.

The next export cannot start until the previous task is COMPLETED **and** its nonempty output files have been verified in Drive. The next source cannot start until all of the current source's EE exports and local uploads are verified. Failures stop the queue. Restarting with unchanged settings resumes progress.

In [ ]:
from pathlib import Path
import os
WORKSPACE = Path(r"D:\Gihub_Projects\BD_GEE_Analysis")
# For Linux/cloud, set WORKSPACE to your actual checkout directory.
if not (WORKSPACE / "catalog.json").is_file():
    raise RuntimeError(f"Repository not found at {WORKSPACE}")
os.chdir(WORKSPACE)
import sys
print("Python:", sys.executable)
print("Workspace:", WORKSPACE)

In [ ]:
# One-time login. Uses your account; no keys belong in this notebook.
from bdgee.core import initialize
initialize(authenticate=True)  # project axial-paratext-491911-b6; EE + Drive scopes

In [ ]:
MODE = "tutorials"  # or "national"; choose one queue per run
THEMES = []          # empty = all; e.g. ["climate", "water"]
START = "2024-01-01" # national mode; tutorial examples retain their own dates
END = "2025-01-01"
from bdgee.batch import main
ARGS = ["--mode", MODE, "--start", START, "--end", END]
if THEMES:
    ARGS += ["--themes", *THEMES]
main(ARGS)  # write/review queue only; no analysis or exports yet

In [ ]:
import json
report = json.loads((WORKSPACE / "outputs" / f"{MODE}_queue.json").read_text())
print("Selected:", len(report["queue"]), "Deferred:", len(report["deferred"]))
for item in report["deferred"]:
    print(item["source"], "—", item["defer_reason"])

In [ ]:
# This cell blocks until completion or a failure. Keep the PC awake and connected.
# Interrupting stops monitoring, but does NOT cancel a submitted EE task.
# Run this same cell again with unchanged settings to resume that task.
main([*ARGS, "--run"])

In [ ]:
state_path = WORKSPACE / "outputs" / f"{MODE}_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text())
    print("Run:", state["phase"])
    for job in state["jobs"]:
        print(job["key"], job["state"], job.get("error", ""))
        for task in job["tasks"]:
            print("  ", task["id"], task["state"], len(task["files"]), "verified files")